In [1]:
# Cell 1) Imports
import re
import urllib.parse
import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 80)
print("[OK] Imports loaded")

[OK] Imports loaded


In [2]:
# Cell 2) DB Config (필요 시 수정)
DB_CONFIG = {
    "host": "100.105.75.47",     # 예: "100.105.75.47"
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

SCHEMA = "c1_fct_detail"
TABLE  = "fct_detail"

def get_engine(cfg: dict):
    user = cfg["user"]
    password = urllib.parse.quote_plus(cfg["password"])
    host = cfg["host"]
    port = cfg["port"]
    dbname = cfg["dbname"]
    conn_str = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
    return create_engine(conn_str, pool_pre_ping=True)

engine = get_engine(DB_CONFIG)
print("[OK] DB engine ready")

[OK] DB engine ready


In [3]:
# Cell 3) 2025-11-01 하루치 + 필요한 contents만 DB에서 로드
# - remark: PD, Non-PD
# - contents: START(대상 step + 경계 step) + '테스트 결과' 라인만 로드(속도 최적화)
# - 정렬: remark, end_day, end_time, test_time

import pandas as pd
from sqlalchemy import text

SCHEMA = "c1_fct_detail"
TABLE  = "fct_detail"

TARGET_DAY = "2025-11-01"   # 하루만

sql = text(f"""
SELECT
    barcode_information,
    remark,
    end_day,        -- DATE 타입
    end_time,       -- TIME 또는 TEXT
    contents,
    test_time       -- HH:MM:SS.ms (실제 시간축)
FROM {SCHEMA}.{TABLE}
WHERE
    end_day = DATE :target_day
    AND remark IN ('PD','Non-PD')
    AND (
        -- PD START (경계 포함)
        contents ILIKE 'START%1.12%'
        OR contents ILIKE 'START%1.13%'
        OR contents ILIKE 'START%1.14%'

        -- Non-PD START (경계 포함)
        OR contents ILIKE 'START%1.22%'
        OR contents ILIKE 'START%1.23%'
        OR contents ILIKE 'START%1.24%'

        -- 블록 종료는 공통적으로 "테스트 결과 : OK/NG/FAIL"
        OR contents ILIKE '%테스트 결과%'
    )
ORDER BY
    remark ASC,
    end_day ASC,
    end_time ASC,
    test_time ASC
""")

df_raw = pd.read_sql(sql, engine, params={"target_day": TARGET_DAY})
print(f"[OK] Loaded rows = {len(df_raw):,} (day={TARGET_DAY})")
df_raw.head(10)

[OK] Loaded rows = 601,175 (day=2025-11-01)


,barcode_information,remark,end_day,end_time,contents,test_time
0,BA1WJ24085501579N1WT-14F014-AC,Non-PD,2025-11-01,08:23:36,테스트 결과 : OK,08:23:15.57
1,BA1WJ24085501579N1WT-14F014-AC,Non-PD,2025-11-01,08:23:36,테스트 결과 : OK,08:23:15.61
2,BA1WJ24085501579N1WT-14F014-AC,Non-PD,2025-11-01,08:23:36,테스트 결과 : OK,08:23:15.77
3,BA1WJ24085501579N1WT-14F014-AC,Non-PD,2025-11-01,08:23:36,테스트 결과 : OK,08:23:15.82
4,BA1WJ24085501579N1WT-14F014-AC,Non-PD,2025-11-01,08:23:36,테스트 결과 : OK,08:23:16.04
5,BA1WJ24085501579N1WT-14F014-AC,Non-PD,2025-11-01,08:23:36,테스트 결과 : OK,08:23:16.28
6,BA1WJ24085501579N1WT-14F014-AC,Non-PD,2025-11-01,08:23:36,테스트 결과 : OK,08:23:16.63
7,BA1WJ24085501579N1WT-14F014-AC,Non-PD,2025-11-01,08:23:36,테스트 결과 : OK,08:23:16.70
8,BA1WJ24085501579N1WT-14F014-AC,Non-PD,2025-11-01,08:23:36,테스트 결과 : OK,08:23:16.74
9,BA1WJ24085501579N1WT-14F014-AC,Non-PD,2025-11-01,08:23:36,테스트 결과 : OK,08:23:16.78


In [4]:
# Cell 4) test_ts 생성(정렬 기준) + PD/Non-PD 분리
# - test_ts = end_day + test_time
# - test_ts가 NaT인 행은 계산 불가이므로 제거

import pandas as pd

def build_test_ts(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    # end_day가 DATE이므로 YYYYMMDD 문자열로 변환
    d["end_day_norm"] = pd.to_datetime(d["end_day"], errors="coerce").dt.strftime("%Y%m%d")

    # test_time까지 결합해서 실제 타임스탬프 생성
    d["test_ts"] = pd.to_datetime(
        d["end_day_norm"].astype(str) + " " + d["test_time"].astype(str),
        errors="coerce"
    )

    # 파싱 실패 제거
    d = d.dropna(subset=["test_ts"]).copy()
    return d

df_raw2 = build_test_ts(df_raw)

df_pd_raw    = df_raw2[df_raw2["remark"] == "PD"].copy()
df_nonpd_raw = df_raw2[df_raw2["remark"] == "Non-PD"].copy()

print("[OK] PD rows   :", len(df_pd_raw))
print("[OK] Non-PD rows:", len(df_nonpd_raw))

[OK] PD rows   : 599799
[OK] Non-PD rows: 1376


In [5]:
# Cell 5) START ↔ 테스트결과 매칭 (검증모드: 원문 그대로 출력)

import re
import pandas as pd

RE_TEST_RESULT = re.compile(r"테스트\s*결과\s*:\s*(OK|NG|FAIL)", re.IGNORECASE)

def calc_test_ct_fullview(df_in, start_steps, next_step_map, max_gap_sec=30):

    d = df_in.copy()
    d = d.sort_values(["barcode_information", "test_ts"]).reset_index(drop=True)

    rows = []

    for bc, g in d.groupby("barcode_information", sort=False):

        g = g.sort_values("test_ts")

        for i, r in g.iterrows():

            # START 라인만 타겟
            if not (str(r["contents"]).startswith("START")):
                continue

            # step 번호 추출
            step = re.search(r"START\s*:+\s*(\d+\.\d+)", r["contents"])
            if not step:
                continue

            step = step.group(1)
            if step not in start_steps:
                continue

            # 다음 step START 찾기
            nxt = g[(g.index > i) & g["contents"].str.contains(f"START :: {next_step_map.get(step,'')}", na=False)]
            if nxt.empty:
                continue

            next_start_idx = nxt.index[0]

            seg = g.loc[(g.index > i) & (g.index < next_start_idx)]
            seg_res = seg[seg["contents"].str.contains("테스트 결과", na=False)]

            if seg_res.empty:
                continue

            last_res = seg_res.iloc[-1]

            test_ct = (last_res["test_ts"] - r["test_ts"]).total_seconds()
            if test_ct < 0 or test_ct > max_gap_sec:
                continue

            rows.append({
                "barcode_information": bc,
                "remark": r["remark"],
                "end_day": r["end_day"],
                "end_time": r["end_time"],
                "start_contents": r["contents"],                  # START 원문
                "start_test_time": r["test_time"],
                "result_contents": last_res["contents"],         # 테스트 결과 원문
                "result_test_time": last_res["test_time"],
                "test_ct": round(test_ct, 3)
            })

    return pd.DataFrame(rows)

In [6]:
# Cell 6) 검증용 출력 (내용 그대로)

pd_ct_full = calc_test_ct_fullview(
    df_pd_raw,
    start_steps={"1.12","1.13"},
    next_step_map={"1.12":"1.13", "1.13":"1.14"},
    max_gap_sec=30
)

nonpd_ct_full = calc_test_ct_fullview(
    df_nonpd_raw,
    start_steps={"1.22","1.23"},
    next_step_map={"1.22":"1.23", "1.23":"1.24"},
    max_gap_sec=30
)

print("========== PD FULL TRACE ==========")
display(pd_ct_full)

print("========== Non-PD FULL TRACE ==========")
display(nonpd_ct_full)

========== PD FULL TRACE ==========


,barcode_information,remark,end_day,end_time,start_contents,start_test_time,result_contents,result_test_time,test_ct
0,BA1WJ24243500234SJ8T-14F014-AE,PD,2025-11-01,08:36:36,START :: 1.12_Test_Carplay_Type-C(B_side),08:36:06.85,테스트 결과 : OK,08:36:08.29,1.44
1,BA1WJ24243500234SJ8T-14F014-AE,PD,2025-11-01,08:36:36,START :: 1.13_Test_Carplay_Type-A,08:36:08.30,테스트 결과 : OK,08:36:09.72,1.42
2,BA1WJ24243500234SJ8T-14F014-AE,PD,2025-11-01,08:41:01,START :: 1.12_Test_Carplay_Type-C(B_side),08:40:44.32,테스트 결과 : OK,08:40:45.75,1.43
3,BA1WJ24243500234SJ8T-14F014-AE,PD,2025-11-01,08:41:01,START :: 1.13_Test_Carplay_Type-A,08:40:45.77,테스트 결과 : OK,08:40:47.20,1.43
4,BA1WJ24243500234SJ8T-14F014-AE,PD,2025-11-01,20:26:12,START :: 1.12_Test_Carplay_Type-C(B_side),20:25:48.97,테스트 결과 : OK,20:25:50.43,1.46
...,...,...,...,...,...,...,...,...,...
14295,BA1WJ25304504283SJ8T-14F014-AF,PD,2025-11-01,01:19:50,START :: 1.13_Test_Carplay_Type-A,01:19:29.85,테스트 결과 : OK,01:19:31.41,1.56
14296,BA1WJ25304504284SJ8T-14F014-AF,PD,2025-11-01,01:19:31,START :: 1.12_Test_Carplay_Type-C(B_side),01:19:13.44,테스트 결과 : OK,01:19:14.94,1.50
14297,BA1WJ25304504284SJ8T-14F014-AF,PD,2025-11-01,01:19:31,START :: 1.13_Test_Carplay_Type-A,01:19:14.95,테스트 결과 : OK,01:19:16.37,1.42
14298,BA1WJ25304504285SJ8T-14F014-AF,PD,2025-11-01,01:19:08,START :: 1.12_Test_Carplay_Type-C(B_side),01:18:44.70,테스트 결과 : OK,01:18:46.13,1.43


========== Non-PD FULL TRACE ==========


,barcode_information,remark,end_day,end_time,start_contents,start_test_time,result_contents,result_test_time,test_ct
0,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-01,08:34:46,START :: 1.22 Test Carplay type-C,08:34:37.10,테스트 결과 : OK,08:34:38.25,1.15
1,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-01,08:34:46,START :: 1.23 Test Carplay usb,08:34:38.26,테스트 결과 : OK,08:34:39.43,1.17
2,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-01,08:39:24,START :: 1.22 Test Carplay type-C,08:39:14.68,테스트 결과 : OK,08:39:15.81,1.13
3,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-01,08:39:24,START :: 1.23 Test Carplay usb,08:39:15.83,테스트 결과 : OK,08:39:16.99,1.16
4,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-01,20:22:57,START :: 1.22 Test Carplay type-C,20:22:48.49,테스트 결과 : OK,20:22:49.64,1.15
5,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-01,20:22:57,START :: 1.23 Test Carplay usb,20:22:49.66,테스트 결과 : OK,20:22:50.85,1.19
6,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-01,20:24:26,START :: 1.22 Test Carplay type-C,20:24:16.68,테스트 결과 : OK,20:24:17.84,1.16
7,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-01,20:24:26,START :: 1.23 Test Carplay usb,20:24:17.86,테스트 결과 : OK,20:24:18.96,1.10
8,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-01,20:31:23,START :: 1.22 Test Carplay type-C,20:31:14.37,테스트 결과 : OK,20:31:15.51,1.14
9,BA1WJ23228500512N1WT-14F014-AC,Non-PD,2025-11-01,20:31:23,START :: 1.23 Test Carplay usb,20:31:15.52,테스트 결과 : OK,20:31:16.69,1.17


In [7]:
# Cell 7) IQR 기반 upper outlier 추출 + upper_outlier_rng(이상치들 min~max)
# 입력: pd_ct_full, nonpd_ct_full
# 출력: pd_upper, nonpd_upper

import pandas as pd
import numpy as np

def get_upper_outliers_with_rng(df_in: pd.DataFrame) -> pd.DataFrame:
    """
    - test_ct로 IQR upper fence 계산
    - upper outlier 행만 반환
    - upper_outlier_rng = (upper outlier들의 test_ct min~max)
    """
    out_cols = [
        "barcode_information", "remark", "end_day", "end_time",
        "start_contents", "result_contents",
        "test_ct", "upper_outlier_rng"
    ]

    if df_in is None or df_in.empty:
        return pd.DataFrame(columns=out_cols)

    d = df_in.copy()
    d["test_ct"] = pd.to_numeric(d["test_ct"], errors="coerce")
    d = d.dropna(subset=["test_ct"]).copy()

    # 표본이 너무 적으면 IQR이 불안정 → 빈 DF
    if len(d) < 8:
        return pd.DataFrame(columns=out_cols)

    q1 = float(d["test_ct"].quantile(0.25))
    q3 = float(d["test_ct"].quantile(0.75))
    iqr = q3 - q1
    upper_fence = q3 + 1.5 * iqr

    out = d[d["test_ct"] > upper_fence].copy()
    if out.empty:
        return pd.DataFrame(columns=out_cols)

    omin = float(out["test_ct"].min())
    omax = float(out["test_ct"].max())
    out["upper_outlier_rng"] = f"{omin:.2f}~{omax:.2f}"

    # 필요한 컬럼만 정리
    for c in ["start_contents", "result_contents"]:
        if c not in out.columns:
            out[c] = np.nan

    out = out[[
        "barcode_information", "remark", "end_day", "end_time",
        "start_contents", "result_contents",
        "test_ct", "upper_outlier_rng"
    ]].sort_values(["end_day","end_time","barcode_information"]).reset_index(drop=True)

    return out

nonpd_upper = get_upper_outliers_with_rng(nonpd_ct_full)
pd_upper    = get_upper_outliers_with_rng(pd_ct_full)

print("========== Non-PD Upper Outliers ==========")
display(nonpd_upper)

print("========== PD Upper Outliers ==========")
display(pd_upper)

========== Non-PD Upper Outliers ==========


,barcode_information,remark,end_day,end_time,start_contents,result_contents,test_ct,upper_outlier_rng


========== PD Upper Outliers ==========


,barcode_information,remark,end_day,end_time,start_contents,result_contents,test_ct,upper_outlier_rng
0,BA1WJ25302500828SJ8T-14F014-AF,PD,2025-11-01,00:02:50,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.61,1.59~7.27
1,BA1WJ25302501402SJ8T-14F014-AF,PD,2025-11-01,00:09:31,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.59,1.59~7.27
2,BA1WJ25303500472SJ8T-14F014-AF,PD,2025-11-01,00:26:07,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.59,1.59~7.27
3,BA1WJ25302501371SJ8T-14F014-AF,PD,2025-11-01,00:31:45,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.59,1.59~7.27
4,BA1WJ25303500024SJ8T-14F014-AF,PD,2025-11-01,00:33:45,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.59,1.59~7.27
5,BA1WJ25303500019SJ8T-14F014-AF,PD,2025-11-01,00:35:19,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.61,1.59~7.27
6,BA1WJ25302501104SJ8T-14F014-AF,PD,2025-11-01,00:50:56,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.59,1.59~7.27
7,BA1WJ25303500450SJ8T-14F014-AF,PD,2025-11-01,00:59:11,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.61,1.59~7.27
8,BA1WJ25302500990SJ8T-14F014-AF,PD,2025-11-01,01:08:03,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.63,1.59~7.27
9,BA1WJ25302500995SJ8T-14F014-AF,PD,2025-11-01,01:14:05,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.59,1.59~7.27


In [9]:
# Cell 8) (수정본) upper outlier 집합에서 barcode별 max_test_ct 기준 Top 5%
# + end_time 다음에 start_contents, result_contents 추가

import math
import pandas as pd
import numpy as np

def top_pct_from_upper_by_max_with_contents(df_upper: pd.DataFrame, pct: float = 0.05) -> pd.DataFrame:
    cols = [
        "barcode_information","remark","end_day","end_time",
        "start_contents","result_contents",
        "max_test_ct","upper_outlier_rng"
    ]

    if df_upper is None or df_upper.empty:
        return pd.DataFrame(columns=cols)

    d = df_upper.copy()
    d["test_ct"] = pd.to_numeric(d["test_ct"], errors="coerce")
    d = d.dropna(subset=["test_ct"]).copy()
    if d.empty:
        return pd.DataFrame(columns=cols)

    # barcode별 최대 이상치값(=max_test_ct)
    agg = (d.groupby(["barcode_information","remark"])["test_ct"]
             .max()
             .reset_index(name="max_test_ct")
             .sort_values("max_test_ct", ascending=False))

    n = max(1, math.ceil(len(agg) * pct))
    top_keys = agg.head(n)[["barcode_information","remark"]]

    # 해당 barcode에서 max가 발생한 행 1개 선택 (동률이면 가장 먼저 나온 것)
    hit = d.merge(top_keys, on=["barcode_information","remark"], how="inner")
    hit = hit.sort_values(["barcode_information","remark","test_ct"], ascending=[True, True, False])

    worst = (hit.drop_duplicates(["barcode_information","remark"], keep="first")
                .rename(columns={"test_ct":"max_test_ct"})
                [[
                    "barcode_information","remark","end_day","end_time",
                    "start_contents","result_contents",
                    "max_test_ct","upper_outlier_rng"
                ]]
                .sort_values("max_test_ct", ascending=False)
                .reset_index(drop=True))

    return worst

# 실행
nonpd_upper_top5 = top_pct_from_upper_by_max_with_contents(nonpd_upper, pct=0.05)
pd_upper_top5    = top_pct_from_upper_by_max_with_contents(pd_upper, pct=0.05)

print("========== Non-PD Upper Outlier Max Top 5% (with contents) ==========")
display(nonpd_upper_top5)

print("========== PD Upper Outlier Max Top 5% (with contents) ==========")
display(pd_upper_top5)

========== Non-PD Upper Outlier Max Top 5% (with contents) ==========


,barcode_information,remark,end_day,end_time,start_contents,result_contents,max_test_ct,upper_outlier_rng


========== PD Upper Outlier Max Top 5% (with contents) ==========


,barcode_information,remark,end_day,end_time,start_contents,result_contents,max_test_ct,upper_outlier_rng
0,BA1WJ25304502945SJ8T-14F014-AF,PD,2025-11-01,12:14:57,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,7.27,1.59~7.27
1,BA1WJ25302500727SJ8T-14F014-AF,PD,2025-11-01,18:40:45,START :: 1.13_Test_Carplay_Type-A,테스트 결과 : OK,3.35,1.59~7.27
2,BA1WJ25302501685SJ8T-14F014-AF,PD,2025-11-01,05:43:55,START :: 1.13_Test_Carplay_Type-A,테스트 결과 : OK,2.29,1.59~7.27
3,BA1WJ25303503456SJ8T-14F014-AF,PD,2025-11-01,20:56:26,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.66,1.59~7.27
4,BA1WJ25302501965SJ8T-14F014-AF,PD,2025-11-01,05:42:26,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.65,1.59~7.27
5,BA1WJ25303503807SJ8T-14F014-AF,PD,2025-11-01,23:10:23,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.65,1.59~7.27
6,BA1WJ25302501447SJ8T-14F014-AF,PD,2025-11-01,02:34:49,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.64,1.59~7.27
7,BA1WJ25303500311SJ8T-14F014-AF,PD,2025-11-01,03:04:43,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.64,1.59~7.27
8,BA1WJ25303503342SJ8T-14F014-AF,PD,2025-11-01,21:36:28,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.64,1.59~7.27
9,BA1WJ25303503813SJ8T-14F014-AF,PD,2025-11-01,18:06:45,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.64,1.59~7.27


In [14]:
# 최종 셀) (중복 완전 제거) Top5 + fct_table(1행 dedup) JOIN
# + 출력 컬럼 순서: barcode_information, station, result, remark, end_day, end_time, run_time, start_contents, result_contents, max_test_ct, upper_outlier_rng

import pandas as pd
from sqlalchemy import text

# -----------------------------
# 1) Top5 합치기 (비어있는 DF 제외하여 FutureWarning 방지)
# -----------------------------
frames = []
if "nonpd_upper_top5" in globals() and nonpd_upper_top5 is not None and not nonpd_upper_top5.empty:
    frames.append(nonpd_upper_top5)
if "pd_upper_top5" in globals() and pd_upper_top5 is not None and not pd_upper_top5.empty:
    frames.append(pd_upper_top5)

if not frames:
    raise ValueError("Top5 데이터가 비어있습니다. (upper outlier가 없었을 가능성)")

df_top5_all = pd.concat(frames, ignore_index=True)

# -----------------------------
# 2) Top5 자체 중복 제거
#    - (barcode_information, end_day, end_time) 동일 조합은 1개만
#    - max_test_ct 큰 것 우선
# -----------------------------
df_top5_all["barcode_information"] = df_top5_all["barcode_information"].astype(str).str.strip()
df_top5_all["max_test_ct"] = pd.to_numeric(df_top5_all["max_test_ct"], errors="coerce")

df_top5_all = (
    df_top5_all
    .sort_values("max_test_ct", ascending=False)
    .drop_duplicates(subset=["barcode_information", "end_day", "end_time"], keep="first")
    .reset_index(drop=True)
)

# -----------------------------
# 3) 조인 키 정규화
#    - fct_table.end_day: TEXT 'YYYYMMDD'
#    - end_time은 숫자만 남긴 end_time_key로 조인
# -----------------------------
d = df_top5_all.copy()

d["end_day_key"] = pd.to_datetime(d["end_day"], errors="coerce").dt.strftime("%Y%m%d")
# 혹시 end_day 파싱 실패하면 숫자만 뽑아서 8자리 사용
mask_bad = d["end_day_key"].isna()
if mask_bad.any():
    d.loc[mask_bad, "end_day_key"] = d.loc[mask_bad, "end_day"].astype(str).str.replace(r"\D","",regex=True).str.slice(0,8)

d["end_time_key"] = d["end_time"].astype(str).str.replace(r"\D", "", regex=True)

barcodes = d["barcode_information"].dropna().astype(str).unique().tolist()
days     = d["end_day_key"].dropna().astype(str).unique().tolist()

# -----------------------------
# 4) fct_table 로드 (매칭 대상만)
# -----------------------------
sql = text("""
SELECT
    barcode_information,
    end_day,              -- TEXT 'YYYYMMDD'
    end_time,
    station,
    run_time,
    result,
    regexp_replace(COALESCE(end_time::text,''), '[^0-9]', '', 'g') AS end_time_key
FROM a2_fct_table.fct_table
WHERE
    barcode_information = ANY(:barcodes)
    AND end_day = ANY(:days)
""")

df_fct = pd.read_sql(sql, engine, params={"barcodes": barcodes, "days": days})

df_fct["barcode_information"] = df_fct["barcode_information"].astype(str).str.strip()
df_fct["end_day_key"] = df_fct["end_day"].astype(str).str.slice(0, 8)
df_fct["end_time_key"] = df_fct["end_time_key"].astype(str)

# -----------------------------
# 5) fct_table 쪽 키 중복 제거(1:N 폭발 방지)
#    - 같은 (barcode, end_day_key, end_time_key) 여러개면 run_time 큰 1개만
# -----------------------------
df_fct["run_time"] = pd.to_numeric(df_fct["run_time"], errors="coerce")

df_fct_dedup = (
    df_fct
    .sort_values(["barcode_information", "end_day_key", "end_time_key", "run_time"],
                 ascending=[True, True, True, False])
    .drop_duplicates(subset=["barcode_information", "end_day_key", "end_time_key"], keep="first")
    .reset_index(drop=True)
)

# -----------------------------
# 6) JOIN
# -----------------------------
merged = d.merge(
    df_fct_dedup[["barcode_information", "end_day_key", "end_time_key", "station", "run_time", "result"]],
    how="left",
    on=["barcode_information", "end_day_key", "end_time_key"]
)

# -----------------------------
# 7) 최종 출력 컬럼 순서 (요청사항 반영)
# -----------------------------
out = (merged[[
        "barcode_information",
        "station",
        "result",
        "remark",
        "end_day",
        "end_time",
        "run_time",
        "start_contents",
        "result_contents",
        "max_test_ct",
        "upper_outlier_rng"
    ]]
    .sort_values("max_test_ct", ascending=False)
    .reset_index(drop=True)
)

print("========== Final Linux Problem Carplay List (Requested Column Order) ==========")
display(out)

========== Final Linux Problem Carplay List (Requested Column Order) ==========


,barcode_information,station,result,remark,end_day,end_time,run_time,start_contents,result_contents,max_test_ct,upper_outlier_rng
0,BA1WJ25304502945SJ8T-14F014-AF,FCT4,PASS,PD,2025-11-01,12:14:57,44.8,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,7.27,1.59~7.27
1,BA1WJ25302500727SJ8T-14F014-AF,FCT2,PASS,PD,2025-11-01,18:40:45,36.6,START :: 1.13_Test_Carplay_Type-A,테스트 결과 : OK,3.35,1.59~7.27
2,BA1WJ25302501685SJ8T-14F014-AF,NaN,NaN,PD,2025-11-01,05:43:55,NaN,START :: 1.13_Test_Carplay_Type-A,테스트 결과 : OK,2.29,1.59~7.27
3,BA1WJ25303503456SJ8T-14F014-AF,FCT1,PASS,PD,2025-11-01,20:56:26,34.8,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.66,1.59~7.27
4,BA1WJ25302501965SJ8T-14F014-AF,NaN,NaN,PD,2025-11-01,05:42:26,NaN,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.65,1.59~7.27
5,BA1WJ25303503807SJ8T-14F014-AF,NaN,NaN,PD,2025-11-01,23:10:23,NaN,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.65,1.59~7.27
6,BA1WJ25302501447SJ8T-14F014-AF,NaN,NaN,PD,2025-11-01,02:34:49,NaN,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.64,1.59~7.27
7,BA1WJ25303500311SJ8T-14F014-AF,NaN,NaN,PD,2025-11-01,03:04:43,NaN,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.64,1.59~7.27
8,BA1WJ25303503342SJ8T-14F014-AF,NaN,NaN,PD,2025-11-01,21:36:28,NaN,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.64,1.59~7.27
9,BA1WJ25303503813SJ8T-14F014-AF,NaN,NaN,PD,2025-11-01,18:06:45,NaN,START :: 1.12_Test_Carplay_Type-C(B_side),테스트 결과 : OK,1.64,1.59~7.27
